# Red neuronal MLP sin escalado

Se reservan los últimos 24 meses para prueba. Los modelos se ajustan solamente con entrenamiento. La caché registra configuraciones y reutiliza resultados si no cambian los datos, parámetros o dependencias.

In [ ]:
%matplotlib inline
import nbimporter
import functions
from pathlib import Path
from forecast_cache import fit_predict_cached, memory

Path("../submission").mkdir(parents=True, exist_ok=True)
df_orig = functions.load_data()
df_orig.head()

In [ ]:
# Los rezagos usan únicamente observaciones anteriores.
config = {"lags": 12, "hidden": 8, "random_state": 123456, "test_months": 24}
x_columns = [f"lagged_{i}m" for i in range(1, config["lags"] + 1)]
df_orig = functions.make_lagged_ts(
    df_orig, p_max=config["lags"], y_column="yt_true", fmt="lagged_{}m"
)
df_dropna = df_orig.dropna(subset=["yt_true"] + x_columns).copy()
df_dropna.head()

Los pronósticos con rezagos se evalúan a un paso: cada mes usa los valores reales anteriores disponibles. No son una proyección recursiva de 24 meses desde un único origen.

In [ ]:
# Partición cronológica; sin mezclar el futuro con entrenamiento.
(X_complete, y_complete, X_train, y_train, X_test, y_test) = functions.train_test_split(
    df=df_dropna, x_columns=x_columns, y_column="yt_true"
)
print(f"Entrenamiento: {len(X_train)} meses; prueba: {len(X_test)} meses")

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import TransformedTargetRegressor

model = MLPRegressor(
    hidden_layer_sizes=(config["hidden"],),
    activation="logistic",
    learning_rate="adaptive",
    momentum=0.01,
    learning_rate_init=0.2,
    max_iter=10000,
    random_state=config["random_state"],
)
pipeline = model

In [ ]:
# Primera ejecución: entrenar y cachear; siguientes: reutilizar si no hay cambios.
prediction_column = f"yt_pred_mlp_{config['hidden']}"
pipeline, predictions = fit_predict_cached(
    pipeline, X_train, y_train, X_complete, config
)
df_orig.loc[X_complete.index, prediction_column] = predictions
df_orig[["yt_true", prediction_column]].tail()

In [ ]:
# Gráfica de los pronósticos
functions.plot_time_series(df=df_orig, yt_col="yt_true")

In [ ]:
# Métricas sobre las filas con pronóstico; se conservan los últimos 24 meses de prueba.
evaluation = df_orig[["yt_true", prediction_column]].dropna()
metrics = memory.cache(functions.compute_evaluation_metrics)(evaluation)
metrics

In [ ]:
# Agregar resultados sin eliminar las columnas de los otros modelos.
functions.save_forecasts(df_orig)
functions.save_metrics(metrics)